# 🔬 Melanoma Skin Cancer Detection
### Binary Classification using EfficientNetB6 Transfer Learning
---

## 1. Import Libraries

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import pathlib
import cv2
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

print('TensorFlow version:', tf.__version__)
print('GPU Available:', tf.config.list_physical_devices('GPU'))

## 2. Install and Configure Kaggle

In [ ]:
!pip install -q kaggle

In [ ]:
from google.colab import files

# Upload your kaggle.json API key
files.upload()

In [ ]:
# Create Kaggle config directory and set permissions
os.makedirs('/root/.kaggle', exist_ok=True)
!mv kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

print('Kaggle configured successfully!')

## 3. Download and Extract Dataset

In [ ]:
!kaggle datasets download -d bhaveshmittal/melanoma-cancer-dataset

In [ ]:
!unzip -q melanoma-cancer-dataset.zip
print('Dataset extracted successfully!')

## 4. Define Dataset Path & Image Parameters

In [ ]:
data_dir = pathlib.Path('/content/train')

img_height  = 180
img_width   = 180
batch_size  = 32

print('Dataset Path:', data_dir)
print('Image Size:  ', img_height, 'x', img_width)
print('Batch Size:  ', batch_size)

## 5. Create Training and Validation Datasets

In [ ]:
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset='training',
    seed=123,
    label_mode='binary',
    image_size=(img_height, img_width),
    batch_size=batch_size
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset='validation',
    seed=123,
    label_mode='binary',
    image_size=(img_height, img_width),
    batch_size=batch_size
)

class_names = train_ds.class_names
print('Classes:', class_names)

## 6. Optimize Dataset Performance (Cache + Shuffle + Prefetch)

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds   = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

print('Datasets optimized with cache + prefetch!')

## 7. Data Augmentation

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
], name='data_augmentation')

print('Data augmentation pipeline ready!')

## 8. Load Pretrained EfficientNetB6

In [ ]:
pretrained_model = tf.keras.applications.EfficientNetB6(
    include_top=False,
    input_shape=(180, 180, 3),
    pooling='avg',
    weights='imagenet'
)

# Freeze all pretrained layers
pretrained_model.trainable = False

print('EfficientNetB6 loaded and frozen!')
print('Total layers:', len(pretrained_model.layers))

## 9. Build the Full Model

In [ ]:
model = Sequential([

    # Data augmentation
    data_augmentation,

    # Normalize pixel values to [0, 1]
    layers.Rescaling(1./255),

    # Pretrained EfficientNetB6 feature extractor
    pretrained_model,

    # Flatten spatial features
    Flatten(),

    # Dense classification head
    Dense(512, activation='relu'),
    Dropout(0.5),

    Dense(256, activation='relu'),
    Dropout(0.3),

    Dense(128, activation='relu'),

    # Binary output
    Dense(1, activation='sigmoid')

], name='melanoma_classifier')

model.summary()

## 10. Compile the Model

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print('Model compiled!')
print('Optimizer : Adam (lr=0.001)')
print('Loss      : Binary Cross-Entropy')
print('Metric    : Accuracy')

## 11. Define Early Stopping Callback

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True,
    verbose=1
)

print('EarlyStopping callback configured (patience=3)')

## 12. Train the Model

In [ ]:
epochs = 10

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=epochs,
    callbacks=[early_stop]
)

## 13. Evaluate Model on Validation Set

In [ ]:
val_loss, val_accuracy = model.evaluate(val_ds)

print(f'\nValidation Loss:     {val_loss:.4f}')
print(f'Validation Accuracy: {val_accuracy * 100:.2f}%')

## 14. Plot Training Accuracy

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(history.history['accuracy'],     label='Train',      color='steelblue', linewidth=2)
plt.plot(history.history['val_accuracy'], label='Validation', color='tomato',    linewidth=2, linestyle='--')

plt.title('Model Accuracy', fontsize=14, fontweight='bold')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 15. Plot Training Loss

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(history.history['loss'],     label='Train',      color='steelblue', linewidth=2)
plt.plot(history.history['val_loss'], label='Validation', color='tomato',    linewidth=2, linestyle='--')

plt.title('Model Loss', fontsize=14, fontweight='bold')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 16. Load and Preprocess a Test Image

In [ ]:
image_path = '/content/test/Benign/6299.jpg'

# Read and convert BGR → RGB
image     = cv2.imread(image_path)
image     = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Resize to model input size
image_resized = cv2.resize(image, (img_height, img_width))

# Add batch dimension: (180,180,3) → (1,180,180,3)
image_input = np.expand_dims(image_resized, axis=0)

print('Image shape after preprocessing:', image_input.shape)

# Display the image
plt.figure(figsize=(4, 4))
plt.imshow(image_resized)
plt.axis('off')
plt.title('Test Image')
plt.show()

## 17. Predict and Display Result

In [ ]:
prediction = model.predict(image_input)
prob = prediction[0][0]

print(f'Raw Prediction Value: {prob:.4f}')

if prob < 0.5:
    label      = 'BENIGN'
    confidence = (1 - prob) * 100
    color      = 'green'
else:
    label      = 'MALIGNANT'
    confidence = prob * 100
    color      = 'red'

print(f'\n>>> Predicted Class : {label}')
print(f'>>> Confidence      : {confidence:.2f}%')

# Display with result overlay
plt.figure(figsize=(4, 4))
plt.imshow(image_resized)
plt.axis('off')
plt.title(f'Prediction: {label}  ({confidence:.1f}%)', color=color, fontsize=13, fontweight='bold')
plt.show()

## 18. Save the Trained Model

In [ ]:
model.save('cancer_model.h5')
print('Model saved successfully as cancer_model.h5')

## 19. (Optional) Load Saved Model

In [ ]:
# Uncomment to reload the model for inference later

# loaded_model = tf.keras.models.load_model('cancer_model.h5')
# print('Model loaded successfully!')
# loaded_model.summary()

---
### ✅ End of Project
**Model:** EfficientNetB6 (frozen) + Custom Dense Head  
**Task:** Binary Classification — Benign vs Malignant  
**Saved:** `cancer_model.h5`